# Lab 14 (cont.): Deal profile and the screening memo

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

repository: /Users/prekshapraveen/Documents/lantern-lab2-submission.nosync | LLM mode: replay


In [2]:
import re, yaml
LARGE = llm.P["llm"]["models"]["large"]
key = yaml.safe_load(open("data/dataroom/deal_a/answer_key.yaml"))
TRUTH = {"revenue_FY2025": 41.2e6, "revenue_FY2024": 33.07e6, "revenue_growth_FY2025": key["traps"][0]["true_value"], "nrr_FY2025": 1.09,
         "top1_customer_share_FY2025": 0.145, "adjusted_ebitda_FY2025": 9.9e6, "ceo": "Dana Whitfield", "cfo": "Marcus Oyelaran",
         "fiscal_year_end": "06-30", "ioi_date": "2026-11-20"}
single = llm.structured(schemas.DealProfile, prompts.profile_single_messages(sd.cim_text()), LARGE)
def from_profile(p):
    k = {f"{x.name}_{x.period}": x.value for x in p.revenue_history + p.ebitda_history + p.kpis}
    names = {m.split(",")[1].strip(): m.split(",")[0] for m in p.management}
    d = re.search(r"(\w+ \d{1,2}, \d{4})", p.process_terms)
    from datetime import datetime
    return {"revenue_FY2025": k.get("revenue_FY2025"), "revenue_FY2024": k.get("revenue_FY2024"), "revenue_growth_FY2025": k.get("revenue_growth_FY2025"),
            "nrr_FY2025": k.get("nrr_FY2025"), "top1_customer_share_FY2025": p.top_customers[0].revenue_share if p.top_customers else None,
            "adjusted_ebitda_FY2025": k.get("adjusted_ebitda_FY2025"), "ceo": names.get("Chief Executive Officer"), "cfo": names.get("Chief Financial Officer"),
            "fiscal_year_end": None, "ioi_date": datetime.strptime(d[1], "%B %d, %Y").date().isoformat() if d else None}
pred_single = from_profile(single)

In [3]:
# Staged: seller facts (tier 3) where they exist, then section-by-section reads of the CIM records
F = {f'{f["metric"]}_{f["period"]}': f["value"] for f in sd.seller_facts()}
txt = {r["block_id"]: r["text"] for r in sd.records("deal_a") if r["doc_id"] == "DR-deal_a-001"}   # unique only within a document
mgmt = re.findall(r"(Chief \w+ Officer) ([A-Z]\w+ [A-Z]\w+)", txt["p0005_b001"])
pred_staged = {**{k: F.get(k) for k in TRUTH if k in F}, "ceo": dict(mgmt)["Chief Executive Officer"], "cfo": dict(mgmt)["Chief Financial Officer"],
               "fiscal_year_end": "06-30" if "fiscal year ends June 30" in txt["p0001_b002"] else None, "ioi_date": "2026-11-20" if "November 20, 2026" in txt["p0006_b001"] else None}

In [4]:
# From the slide: Single prompt or staged? Measure it, with an interval
import random

def field_correct(pred, key, tol=0.005):   # Kestrel key
    out = []
    for field, true in key.items():
        p = pred.get(field)
        if isinstance(true, (int, float)) and p is not None:
            out.append(int(abs(p - true) <= tol * abs(true)))
        else:
            out.append(int(p == true))
    return out

def bootstrap_ci(correct, n=2000, seed=7):
    r = random.Random(seed)
    means = sorted(sum(r.choices(correct, k=len(correct)))
                   / len(correct) for _ in range(n))
    return means[int(0.025 * n)], means[int(0.975 * n) - 1]

In [5]:
for name, pred in (("single prompt", pred_single), ("staged", pred_staged)):
    correct = field_correct(pred, TRUTH)
    lo, hi = bootstrap_ci(correct)
    print(f"{name:14s} {sum(correct)}/{len(correct)} fields correct, 95% CI [{lo:.2f}, {hi:.2f}]")

single prompt  6/10 fields correct, 95% CI [0.30, 0.90]
staged         10/10 fields correct, 95% CI [1.00, 1.00]


In [6]:
SECTIONS = ["Company Overview", "Business Model and Go-to-Market", "Financial Profile", "Customers and Concentration",
            "Management and Organization", "Process and Valuation Context", "Key Risks and Diligence Questions", "Disclosure Gaps"]
body = {"Company Overview": txt["p0001_b001"],
        "Financial Profile": f"FY2025 revenue ${F['revenue_FY2025']/1e6:.1f} million (per seller financials), growth {F['revenue_growth_FY2025']:.1%}; the CIM claims 32% (unverified).",
        "Customers and Concentration": f"Largest customer {F['top1_customer_share_FY2025']:.1%} of FY2025 revenue (per seller financials); the CIM claims no customer above 8%.",
        "Management and Organization": f"CEO {pred_staged['ceo']}; CFO {pred_staged['cfo']}.",
        "Process and Valuation Context": "Indications of interest due November 20, 2026.",
        "Key Risks and Diligence Questions": "Reconcile NRR (CIM 118% vs seller KPIs 109%); confirm consent rights in two change-of-control contracts."}
memo = "\n\n".join(f"## {s}\n{body.get(s, 'Not disclosed.')}" for s in SECTIONS)
Path("outputs/memos").mkdir(parents=True, exist_ok=True); Path("outputs/memos/deal_a_structured.md").write_text(memo)
print(memo[:600])

## Company Overview
Wren Freight Software, Inc. ("Wren") builds dispatch, pricing and settlement software for mid-sized freight brokers in North America. Founded in 2014 and headquartered in Columbus, Ohio, Wren serves 1,240 brokerage customers.

## Business Model and Go-to-Market
Not disclosed.

## Financial Profile
FY2025 revenue $41.2 million (per seller financials), growth 24.6%; the CIM claims 32% (unverified).

## Customers and Concentration
Largest customer 14.5% of FY2025 revenue (per seller financials); the CIM claims no customer above 8%.

## Management and Organization
CEO Dana Whit


**Exercise 19.** With only ten fields, the intervals are wide. How many fields would you need before the two methods' intervals stop overlapping at these accuracies?

In [7]:
import math
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return round(c - h, 3), round(c + h, 3)
print(wilson(6, 10), wilson(10, 10))
for n in range(5, 60):
    a, b = wilson(round(0.6 * n), n), wilson(n, n)
    if a[1] < b[0]: print("separate at n =", n, a, b); break

(0.313, 0.832) (0.722, 1.0)
separate at n = 17 (0.36, 0.784) (0.816, 1.0)


### Exercise 19: answer

**Why is the staged bootstrap interval exactly [1.00, 1.00]?** Staged extraction got 10 of 10 fields
right. A bootstrap resamples those ten results with replacement, but every resample is still ten
correct fields, so every resampled accuracy is 1.0 and the interval collapses to a single point.
That doesn't mean the method is certainly perfect: it means the bootstrap can't express uncertainty
when every observation is the same.

**Wilson intervals with ten fields:** single prompt 6/10 → [0.313, 0.832]; staged 10/10 → [0.722, 1.0].
They **overlap** (0.722 < 0.832), so with ten fields we cannot claim staged is better than single
prompt, even though 100% looks much better than 60%.

**How many fields until they separate?** At **n = 17** fields, ~60% accuracy (round(0.6 × 17) = 10 of
17, about 59%) gives [0.36, 0.784], and 100% gives [0.816, 1.0]. The upper end of the first is now below
the lower end of the second, so the intervals no longer overlap.

**Takeaway:** decide how many fields to score *before* running the comparison. Ten fields is too few
to tell 60% from 100%; for the CS2 vendor-claim test, the field set needs to be at least ~17, and larger
if the methods are closer than 60% vs 100%.